# DA6401 Assignment 1 - MLP W&B Report

Weights & Biases experiments and analysis for the Multi-Layer Perceptron assignment.

## Setup

In [1]:
from dotenv import load_dotenv
import os
import sys

# Load .env from project root (works whether cwd is project root or notebooks/)
for d in [os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))]:
    env_path = os.path.join(d, '.env')
    if os.path.exists(env_path):
        load_dotenv(env_path)
        break
else:
    load_dotenv()

import numpy as np
import wandb
import matplotlib.pyplot as plt
from keras.datasets import mnist, fashion_mnist

# Add project root to path (src.* imports need project root)
project_root = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

## 2.1 Data Exploration and Class Distribution (3 Marks)

Log a W&B Table containing 5 sample images from each of the 10 classes in the dataset.
Identify any classes that look visually similar in their raw form. How might this visual similarity impact your model?

In [2]:
def load_dataset(dataset_name='mnist'):
    """Load MNIST or Fashion-MNIST."""
    if dataset_name == 'mnist':
        (X_train, y_train), (X_test, y_test) = mnist.load_data()
    else:
        (X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
    return X_train, y_train, X_test, y_test

def get_samples_per_class(X, y, samples_per_class=5):
    """Get samples_per_class images for each of the 10 classes."""
    images, labels = [], []
    for c in range(10):
        idx = np.where(y == c)[0][:samples_per_class]
        for i in idx:
            images.append(X[i])
            labels.append(c)
    return np.array(images), np.array(labels)

In [3]:
MNIST_LABELS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
FASHION_LABELS = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def log_data_exploration_table(dataset_name='mnist', project='da6401_assignment1'):
    """Log 5 sample images per class to W&B Table."""
    X_train, y_train, _, _ = load_dataset(dataset_name)
    images, labels = get_samples_per_class(X_train, y_train, samples_per_class=5)
    
    labels_names = MNIST_LABELS if dataset_name == 'mnist' else FASHION_LABELS
    
    wandb.init(project=project, name=f'data_exploration_{dataset_name}', job_type='data_exploration')
    
    table_data = []
    for i in range(len(images)):
        img = images[i]
        label = labels[i]
        table_data.append([
            wandb.Image(img, caption=f'Class {label} ({labels_names[label]})'),
            label,
            labels_names[label]
        ])
    
    table = wandb.Table(columns=['image', 'class', 'class_name'], data=table_data)
    wandb.log({'sample_images_per_class': table})
    wandb.finish()
    print(f'Logged {len(table_data)} images for {dataset_name}')

# Run for MNIST
log_data_exploration_table('mnist')

# Run for Fashion-MNIST (optional)
# log_data_exploration_table('fashion_mnist')

wandb: Currently logged in as: ma24m011 (ma24m011-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged 50 images for mnist


## 2.2 Hyperparameter Sweep (6 Marks)

Perform a W&B Sweep with at least 100 runs, varying hyperparameters. Using the Parallel Coordinates plot, identify which hyperparameter had the most significant impact on validation accuracy. What was your best-performing configuration?

In [ ]:
# Ensure project root is in path (run Setup cell first, or this will fix it)
import os, sys
_project_root = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.utils.data_loader import load_dataset as load_dataset_flat, one_hot_encode
from src.ann.neural_network import NeuralNetwork
from sklearn.model_selection import train_test_split
import argparse

def sweep_train():
    """Training function for W&B sweep. Uses wandb.config for hyperparameters."""
    with wandb.init() as run:
        config = wandb.config
        
        # Build args from sweep config
        args = argparse.Namespace(
            dataset='mnist',
            epochs=5,  # Short for sweep (100 runs)
            batch_size=config.batch_size,
            learning_rate=config.learning_rate,
            optimizer=config.optimizer,
            num_layers=config.num_layers,
            hidden_size=[config.hidden_size] * config.num_layers,
            hidden_layers=[config.hidden_size] * config.num_layers,
            activation=config.activation,
            loss=config.loss,
            weight_init=config.weight_init,
            weight_decay=0.0,
            wandb_project='da6401_assignment1',
        )
        
        X_train, y_train, X_test, y_test = load_dataset_flat(args.dataset)
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
        
        model = NeuralNetwork(args)
        best_val_f1 = 0.0
        best_weights = model.get_weights()  # fallback if no improvement
        
        for ep in range(args.epochs):
            avg_loss = model.train(X_tr, y_tr, epochs=1, batch_size=args.batch_size)
            val_result = model.evaluate(X_val, y_val)
            tr_result = model.evaluate(X_tr, y_tr)
            
            wandb.log({
                'epoch': ep + 1,
                'train_loss': avg_loss,
                'train_accuracy': tr_result['accuracy'],
                'val_accuracy': val_result['accuracy'],
                'val_f1': val_result['f1'],
            })
            
            if val_result['f1'] > best_val_f1:
                best_val_f1 = val_result['f1']
                best_weights = model.get_weights()
        
        model.set_weights(best_weights)
        test_result = model.evaluate(X_test, y_test)
        wandb.log({
            'test_accuracy': test_result['accuracy'],
            'test_f1': test_result['f1'],
            'val_accuracy_final': best_val_f1,
        })

In [ ]:
# Sweep configuration - at least 100 runs
sweep_config = {
    'method': 'random',  # random search for 100 runs
    'metric': {'name': 'val_accuracy_final', 'goal': 'maximize'},
    'parameters': {
        'learning_rate': {'values': [0.0001, 0.001, 0.01, 0.1]},
        'batch_size': {'values': [32, 64, 128]},
        'optimizer': {'values': ['sgd', 'momentum', 'rmsprop']},
        'num_layers': {'values': [1, 2, 3]},
        'hidden_size': {'values': [64, 128]},
        'activation': {'values': ['relu', 'sigmoid', 'tanh']},
        'weight_init': {'values': ['random', 'xavier']},
        'loss': {'values': ['cross_entropy', 'mse']},
    }
}

sweep_id = wandb.sweep(sweep=sweep_config, project='da6401_assignment1')
print(f'Sweep ID: {sweep_id}')

Create sweep with ID: y7zzl04z
Sweep URL: https://wandb.ai/ma24m011-iit-madras/da6401_assignment1/sweeps/y7zzl04z
Sweep ID: y7zzl04z


In [ ]:
# Run the sweep - 100 runs (takes a while!)
# Each run: 5 epochs on 80% of MNIST train, validate on 20%
wandb.agent(sweep_id, function=sweep_train, count=100)

wandb: Agent Starting Run: v1bnqwyk with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 32
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.01
wandb: 	loss: mse
wandb: 	num_layers: 3
wandb: 	optimizer: rmsprop
wandb: 	weight_init: random
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
train_accuracy,▁▆▇██
train_loss,█▂▁▁▁
val_accuracy,▁▆▇██
val_accuracy_final,▁
val_f1,▁▆▇██
epoch,5
test_accuracy,0.9399
test_f1,0.93948


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: lcd7vv60 with config:
wandb: 	activation: tanh
wandb: 	batch_size: 32
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.1
wandb: 	loss: mse
wandb: 	num_layers: 2
wandb: 	optimizer: rmsprop
wandb: 	weight_init: random
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
train_accuracy,█▁▁▁▁
train_loss,█▁▁▁▁
val_accuracy,█▁▁▁▁
val_accuracy_final,▁
val_f1,█▁▁▁▁
epoch,5
test_accuracy,0.101
test_f1,0.01835


wandb: Agent Starting Run: 9eerc3nh with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 128
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.001
wandb: 	loss: mse
wandb: 	num_layers: 1
wandb: 	optimizer: rmsprop
wandb: 	weight_init: xavier
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


...

wandb: Agent Starting Run: 4anz3ae5 with config:
wandb: 	activation: relu
wandb: 	batch_size: 64
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.01
wandb: 	loss: mse
wandb: 	num_layers: 3
wandb: 	optimizer: sgd
wandb: 	weight_init: random
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\neural_layer.py:89: RuntimeWarning: invalid value encountered in multiply
  return grad_from_next*relu_grad(self.input)
c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\optimizers.py:15: RuntimeWarning: invalid value encountered in multiply
  grad_W = layer.grad_W+self.wd * layer.W


epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
train_accuracy,▁▁▁▁▁
val_accuracy,▁▁▁▁▁
val_accuracy_final,▁
val_f1,▁▁▁▁▁
epoch,5
test_accuracy,0.098
test_f1,0.01785
train_accuracy,0.09892


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: wpffhdu5 with config:
wandb: 	activation: tanh
wandb: 	batch_size: 64
wandb: 	hidden_size: 64
wandb: 	learning_rate: 0.01
wandb: 	loss: mse
wandb: 	num_layers: 2
wandb: 	optimizer: sgd
wandb: 	weight_init: random
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


epoch,▁▃▅▆█
test_accuracy,▁
test_f1,▁
train_accuracy,▃▁▃▆█
train_loss,█▄▂▁▁
val_accuracy,█▁▃▆▆
val_accuracy_final,▁
val_f1,▇▁▄▇█
epoch,5
test_accuracy,0.1213
test_f1,0.11987


## 2.3 The Optimizer Showdown (5 Marks)

Compare the convergence rates of all optimizers (sgd, momentum, rmsprop) using the same architecture: **3 hidden layers, 128 neurons each, ReLU activation**. Which optimizer minimized the loss fastest in the first 5 epochs? Theoretically, why does RMSProp often outperform standard SGD on image classification?

In [4]:
# Fixed architecture: 3 hidden layers, 128 neurons each, ReLU
FIXED_CONFIG = {
    'dataset': 'mnist',
    'epochs': 5,
    'batch_size': 64,
    'learning_rate': 0.01,
    'num_layers': 3,
    'hidden_size': [128, 128, 128],
    'activation': 'relu',
    'loss': 'cross_entropy',
    'weight_init': 'xavier',
    'weight_decay': 0.0,
}

def run_optimizer_comparison(optimizers=['sgd', 'momentum', 'rmsprop']):
    """Run training with each optimizer and log to W&B for comparison."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    X_train, y_train, X_test, y_test = load_dataset('mnist')
    
    for opt in optimizers:
        cfg = {**FIXED_CONFIG, 'optimizer': opt}
        args = argparse.Namespace(**cfg)
        args.hidden_layers = args.hidden_size
        
        run = wandb.init(project='da6401_assignment1', name=f'optimizer_{opt}', config=cfg)
        model = NeuralNetwork(args)
        
        for ep in range(args.epochs):
            avg_loss = model.train(X_train, y_train, epochs=1, batch_size=args.batch_size)
            result = model.evaluate(X_test, y_test)
            wandb.log({'epoch': ep+1, 'train_loss': avg_loss, 'test_accuracy': result['accuracy']})
        
        wandb.finish()
        print(f'{opt}: final loss={avg_loss:.4f}, accuracy={result["accuracy"]:.4f}')

In [5]:
# Run all 3 optimizers - each logs train_loss per epoch to W&B
# In W&B: Charts > Add panel > Line plot > X: epoch, Y: train_loss, Group by: run
run_optimizer_comparison(['sgd', 'momentum', 'rmsprop'])

epoch,▁▃▅▆█
test_accuracy,▁▂▄▆█
train_loss,█▆▅▃▁
epoch,5
test_accuracy,0.2515
train_loss,2.25451


sgd: final loss=2.2545, accuracy=0.2515


epoch,▁▃▅▆█
test_accuracy,▁▅▆▇█
train_loss,█▆▃▂▁
epoch,5
test_accuracy,0.8634
train_loss,0.58659


momentum: final loss=0.5866, accuracy=0.8634


epoch,▁▃▅▆█
test_accuracy,▁▇▃▇█
train_loss,█▂▂▁▁
epoch,5
test_accuracy,0.9678
train_loss,0.11775


rmsprop: final loss=0.1178, accuracy=0.9678


**For your W&B report:** In W&B Charts, create a line plot with X=epoch, Y=train_loss, grouped by run name. Compare the curves. Then write: (1) Which optimizer reduced loss fastest in the first 5 epochs? (2) Why does RMSProp often outperform SGD on image classification? (Adaptive per-parameter learning rates; handles varying gradient scales across layers.)

## 2.4 Vanishing Gradient Analysis (5 Marks)

Fix the optimizer to RMSProp and compare Sigmoid and ReLU for different network configurations. Log the gradient norms for the first hidden layer. Do you observe the vanishing gradient problem with Sigmoid? Provide a plot to support your observation.

In [6]:
def run_vanishing_gradient_analysis(epochs=10, batch_size=64):
    """Compare Sigmoid vs ReLU: log first hidden layer gradient norms. Use RMSProp."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    X_train, y_train, X_test, y_test = load_dataset('mnist')
    Y_train = one_hot_encode(y_train).T
    N = X_train.shape[0]
    
    configs = [
        {'name': 'sigmoid_2layers', 'activation': 'sigmoid', 'hidden_size': [128, 128]},
        {'name': 'relu_2layers', 'activation': 'relu', 'hidden_size': [128, 128]},
        {'name': 'sigmoid_4layers', 'activation': 'sigmoid', 'hidden_size': [128, 128, 128, 128]},
        {'name': 'relu_4layers', 'activation': 'relu', 'hidden_size': [128, 128, 128, 128]},
    ]
    
    for cfg in configs:
        args = argparse.Namespace(
            dataset='mnist', optimizer='rmsprop', learning_rate=0.001,
            num_layers=len(cfg['hidden_size']), hidden_size=cfg['hidden_size'],
            hidden_layers=cfg['hidden_size'], activation=cfg['activation'],
            loss='cross_entropy', weight_init='xavier', weight_decay=0.0,
        )
        run = wandb.init(project='da6401_assignment1', name=f'vanishing_{cfg["name"]}', config=cfg)
        model = NeuralNetwork(args)
        
        for ep in range(epochs):
            perm = np.random.permutation(N)
            total_loss, total_grad_norm, count = 0.0, 0.0, 0
            for start in range(0, N, batch_size):
                end = min(start + batch_size, N)
                X_b = X_train[perm[start:end]]
                Y_b = Y_train[:, perm[start:end]]
                y_hat = model.forward(X_b)
                _, _, loss = model.backward(Y_b.T, y_hat)
                grad_norm = np.linalg.norm(model.param_layers[0].grad_W)  # first hidden layer
                total_loss += loss
                total_grad_norm += grad_norm
                count += 1
                model.update_weights()
            
            wandb.log({
                'epoch': ep + 1,
                'train_loss': total_loss / count,
                'first_layer_grad_norm': total_grad_norm / count,
            })
        
        wandb.finish()
        print(f'{cfg["name"]}: done')

**For your W&B report:** Create a line plot with X=epoch, Y=first_layer_grad_norm, Group by run. Sigmoid runs should show gradient norms dropping toward zero (vanishing). ReLU runs should keep gradients more stable. Use this plot to support your observation about the vanishing gradient problem.

In [7]:
# Run vanishing gradient analysis
run_vanishing_gradient_analysis(epochs=10)

epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,▁▆██████▇▇
train_loss,█▃▂▂▂▁▁▁▁▁
epoch,10
first_layer_grad_norm,0.00412
train_loss,0.16122


sigmoid_2layers: done


epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,█▆▅▄▄▃▃▂▂▁
train_loss,█▄▃▂▂▂▁▁▁▁
epoch,10
first_layer_grad_norm,0.00726
train_loss,0.02637


relu_2layers: done


epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,▁▁▂▅▆▇████
train_loss,██▇▄▂▂▂▁▁▁
epoch,10
first_layer_grad_norm,0.00809
train_loss,0.25328


sigmoid_4layers: done


epoch,▁▂▃▃▄▅▆▆▇█
first_layer_grad_norm,█▆▅▄▄▃▃▂▂▁
train_loss,█▃▃▂▂▂▁▁▁▁
epoch,10
first_layer_grad_norm,0.00851
train_loss,0.02207


relu_4layers: done


## 2.5 The "Dead Neuron" Investigation (6 Marks)

Using ReLU + high learning rate (0.1), monitor activations of hidden layers. Find a run where validation accuracy plateaus early. Look at activation distribution. Identify dead neurons (output zero for all inputs). Compare with Tanh run.

In [9]:
def get_hidden_activations_and_output(model, X):
    """Run forward, collect hidden activations and return (activations_list, output_logits)."""
    from src.ann.neural_layer import ActivationLayer
    a = X.T
    activations = []
    for layer in model.layers:
        a = layer.forward_pass(a)
        if type(layer) == ActivationLayer and len(activations) < len(model.param_layers) - 1:
            activations.append(a.copy())
    return activations, a.T

def run_dead_neuron_analysis(epochs=15, batch_size=64):
    """ReLU + high LR (0.1) vs Tanh. Log activations and dead neuron count."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    from sklearn.model_selection import train_test_split
    import argparse
    
    X_train, y_train, X_test, y_test = load_dataset('mnist')
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
    Y_tr = one_hot_encode(y_tr).T
    N = X_tr.shape[0]
    
    for act_name, lr in [('relu_high_lr', 0.1), ('tanh_high_lr', 0.1)]:
        args = argparse.Namespace(dataset='mnist', optimizer='rmsprop', learning_rate=lr,
            num_layers=3, hidden_size=[128,128,128], hidden_layers=[128,128,128],
            activation=act_name.split('_')[0], loss='cross_entropy', weight_init='xavier', weight_decay=0.0)
        run = wandb.init(project='da6401_assignment1', name=f'dead_neuron_{act_name}', config={'activation': act_name.split('_')[0], 'lr': lr})
        model = NeuralNetwork(args)
        
        for ep in range(epochs):
            perm = np.random.permutation(N)
            total_loss, total_dead_pct, count = 0.0, 0.0, 0
            for start in range(0, N, batch_size):
                end = min(start + batch_size, N)
                X_b = X_tr[perm[start:end]]
                Y_b = Y_tr[:, perm[start:end]]
                acts, y_hat = get_hidden_activations_and_output(model, X_b)
                dead_count = sum(np.sum(np.all(a == 0, axis=1)) for a in acts)
                total_neurons = sum(a.shape[0] for a in acts)
                total_dead_pct += 100 * dead_count / total_neurons if total_neurons > 0 else 0
                _, _, loss = model.backward(Y_b.T, y_hat)
                total_loss += loss
                count += 1
                model.update_weights()
            val_res = model.evaluate(X_val, y_val)
            wandb.log({'epoch': ep+1, 'train_loss': total_loss/count, 'val_accuracy': val_res['accuracy'],
                       'dead_neuron_pct': total_dead_pct/count})
        wandb.finish()
        print(f'{act_name}: done')

In [10]:
run_dead_neuron_analysis(epochs=15)


c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\objective_functions.py:24: RuntimeWarning: overflow encountered in exp
  exp_pred = np.exp(y_hat)
c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\objective_functions.py:31: RuntimeWarning: invalid value encountered in scalar divide
  probabilities[i, j] =exp_pred[i, j]/col_sum


dead_neuron_pct,▁██████████████
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dead_neuron_pct,100
epoch,15
train_loss,nan
val_accuracy,0.09792


relu_high_lr: done


c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\objective_functions.py:24: RuntimeWarning: overflow encountered in exp
  exp_pred = np.exp(y_hat)
c:\Users\jeets\Cursor_files\Deep_learning_ass1\src\ann\objective_functions.py:31: RuntimeWarning: invalid value encountered in scalar divide
  probabilities[i, j] =exp_pred[i, j]/col_sum


dead_neuron_pct,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▂▁▂▄▃▂▁
val_accuracy,█▂▁▁▂█▁▂▂▂▂▂▂▂▂
dead_neuron_pct,0
epoch,15
train_loss,nan
val_accuracy,0.09792


tanh_high_lr: done


## 2.6 Loss Function Comparison (4 Marks)

Compare MSE vs Cross-Entropy. Same architecture and learning rate. Which converged faster? Why is Cross-Entropy better suited for multi-class classification with Softmax?

In [11]:
def run_loss_comparison(epochs=10, batch_size=64):
    """Compare MSE vs Cross-Entropy. Same arch: 3 hidden, 128 each, ReLU, lr=0.001."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    X_train, y_train, X_test, y_test = load_dataset('mnist')
    base = {'dataset':'mnist','optimizer':'rmsprop','learning_rate':0.001,'num_layers':3,
            'hidden_size':[128,128,128],'hidden_layers':[128,128,128],'activation':'relu',
            'weight_init':'xavier','weight_decay':0.0}
    for loss in ['mse', 'cross_entropy']:
        args = argparse.Namespace(**{**base, 'loss': loss})
        run = wandb.init(project='da6401_assignment1', name=f'loss_{loss}', config={'loss': loss})
        model = NeuralNetwork(args)
        for ep in range(epochs):
            avg_loss = model.train(X_train, y_train, epochs=1, batch_size=batch_size)
            res = model.evaluate(X_test, y_test)
            wandb.log({'epoch': ep+1, 'train_loss': avg_loss, 'test_accuracy': res['accuracy']})
        wandb.finish()
        print(f'{loss}: done')

In [12]:
run_loss_comparison(epochs=10)

epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁▃▆▃▇▇████
train_loss,█▄▃▂▂▂▁▁▁▁
epoch,10
test_accuracy,0.9777
train_loss,0.0227


mse: done


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁▇█▇█████▇
train_loss,█▃▃▂▂▂▁▁▁▁
epoch,10
test_accuracy,0.9649
train_loss,0.02095


cross_entropy: done


## 2.7 Global Performance Analysis (4 Marks)

Create an overlay plot of Training vs Test Accuracy across every run from your hyperparameter search (Section 2.2). Identify runs with high training but poor test accuracy. What does this gap indicate? (Overfitting.)

In [ ]:
# No code needed - use W&B Charts on your sweep data (Section 2.2)
# Add panel: Line plot, X=epoch, Y1=train_accuracy, Y2=val_accuracy (or test), Group by run
# Filter to sweep runs. Identify runs where train >> test (overfitting).

## 2.8 Error Analysis (5 Marks)

Plot Confusion Matrix for best model on test set. For more marks: creative visualization of model failures.

In [13]:
def run_error_analysis(model_path='src/best_model.npy', config_path='src/best_model_config.json'):
    """Load best model, plot confusion matrix, log to W&B."""
    import os, sys, json
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    model_path = os.path.join(_pr, model_path) if not os.path.isabs(model_path) else model_path
    config_path = os.path.join(_pr, config_path) if not os.path.isabs(config_path) else config_path
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    from src.utils.data_loader import load_dataset
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    if not os.path.exists(model_path) or not os.path.exists(config_path):
        print('Run training first to create best_model.npy and best_model_config.json')
        return
    weights = np.load(model_path, allow_pickle=True).item()
    with open(config_path) as f:
        cfg = json.load(f)
    args = argparse.Namespace(**cfg)
    args.hidden_layers = args.hidden_size
    model = NeuralNetwork(args)
    model.set_weights(weights)
    X_train, y_train, X_test, y_test = load_dataset(args.dataset)
    res = model.evaluate(X_test, y_test)
    cm = res['confusion_matrix']
    
    run = wandb.init(project='da6401_assignment1', name='error_analysis')
    
    plt.figure(figsize=(10,8))
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title('Confusion Matrix')
    for i in range(10):
        for j in range(10):
            plt.text(j, i, str(cm[i,j]), ha='center', va='center')
    plt.savefig('confusion_matrix.png')
    wandb.log({'confusion_matrix_img': wandb.Image('confusion_matrix.png')})
    plt.close()
    wandb.finish()
    print(f'Accuracy: {res["accuracy"]:.4f}')

In [14]:
run_error_analysis()

Accuracy: 0.9778


## 2.9 Weight Initialization & Symmetry (7 Marks)

Compare Zeros vs Xavier init. Log gradients of 5 neurons in first hidden layer over first 50 iterations. In Zeros run, gradients are identical (symmetry). Explain why symmetry breaking is necessary.

In [15]:
def run_weight_init_comparison(iterations=50, batch_size=64):
    """Zeros vs Xavier. Log gradients of 5 neurons in first hidden layer."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    X_train, y_train, _, _ = load_dataset('mnist')
    Y_train = one_hot_encode(y_train).T
    N = X_train.shape[0]
    
    for init_name in ['zeros', 'xavier']:
        args = argparse.Namespace(dataset='mnist', optimizer='sgd', learning_rate=0.01,
            num_layers=2, hidden_size=[64, 64], hidden_layers=[64, 64],
            activation='relu', loss='cross_entropy', weight_init=init_name, weight_decay=0.0)
        run = wandb.init(project='da6401_assignment1', name=f'weight_init_{init_name}', config={'weight_init': init_name})
        model = NeuralNetwork(args)
        perm = np.random.permutation(N)
        for it in range(iterations):
            start = (it * batch_size) % N
            end = min(start + batch_size, N)
            if start >= end:
                perm = np.random.permutation(N)
                start, end = 0, batch_size
            X_b = X_train[perm[start:end]]
            Y_b = Y_train[:, perm[start:end]]
            y_hat = model.forward(X_b)
            _, _, loss = model.backward(Y_b.T, y_hat)
            grad_W0 = model.param_layers[0].grad_W  # (64, 784)
            neuron_grads = [np.linalg.norm(grad_W0[i, :]) for i in range(5)]
            wandb.log({'iteration': it, 'loss': loss, **{f'neuron_{i}_grad_norm': neuron_grads[i] for i in range(5)}})
            model.update_weights()
        wandb.finish()
        print(f'{init_name}: done')

In [16]:
run_weight_init_comparison(iterations=50)

iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,▆▆▆▆▅▅▆▇▆▅▆▅▇▇▅▆▆▆▆▇▅▆▇▃▆▆▆▅▄█▆▃▄▇▅▅▅▅▆▁
neuron_0_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_1_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_3_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_4_grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iteration,49
loss,2.30257
neuron_0_grad_norm,0
neuron_1_grad_norm,0


zeros: done


iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
loss,▅▅▄▆▅▆▆▄▆▅▇▆▄▁▅▂▄█▄▅▄▄▄▂█▅▄▄▆▃▆▅▂▅▃▆▆▂▆▅
neuron_0_grad_norm,▂▇▄▂▃▃▅▃▅█▅▃▁▃▂▄▆▅▃▃▁▃▇▄▃▂▃▄▄▅▂▄▄▄▃▅▇▅▄▆
neuron_1_grad_norm,▃▁▂▅▃▄▃▇▅▄▇▃▂▂▂▅▂▅▆▆▄▂▅▄▄▂▃▅▂▄▃▂█▃▅▂▁▂▂▄
neuron_2_grad_norm,▁▅▆▃▆█▄▄▃▂▅▆▄▄▅▂▄█▇▂▄▆▄▂▃▂▇▂▃█▄▄▃▇▂▄▄▁▂▂
neuron_3_grad_norm,▁▃▅▆▄▃▅▃▄█▄▂▃▃▄▇▅▁▇▃▁▄▅▂▂▃▃▇▃█▆▄▃▆▄▃▆▁▃▄
neuron_4_grad_norm,█▅▂▁▇▂▃▃▅▃▂▃▂▂▃▂▃▄▅▃▄▁▂▄▃▂▂▇▃▃▃▂▁▂▂▂▂▂▄▂
iteration,49
loss,2.33075
neuron_0_grad_norm,0.00266
neuron_1_grad_norm,0.00191


xavier: done


## 2.10 The Fashion-MNIST Transfer Challenge (5 Marks)

Only 3 hyperparameter configs allowed. Based on MNIST learnings, choose 3 (Architecture + Optimizer + Activation). Report accuracies. Did best-for-digits work best for clothing?

In [17]:
def run_fashion_mnist_3configs(epochs=15, batch_size=64):
    """3 configs on Fashion-MNIST. Based on MNIST: rmsprop+relu often best."""
    import os, sys
    _pr = os.getcwd() if os.path.exists(os.path.join(os.getcwd(), 'src')) else os.path.abspath(os.path.join(os.getcwd(), '..'))
    if _pr not in sys.path:
        sys.path.insert(0, _pr)
    from src.utils.data_loader import load_dataset, one_hot_encode
    from src.ann.neural_network import NeuralNetwork
    import argparse
    
    X_train, y_train, X_test, y_test = load_dataset('fashion_mnist')
    configs = [
        {'name': 'rmsprop_relu_3x128', 'optimizer': 'rmsprop', 'activation': 'relu', 'hidden_size': [128, 128, 128]},
        {'name': 'momentum_relu_3x128', 'optimizer': 'momentum', 'activation': 'relu', 'hidden_size': [128, 128, 128]},
        {'name': 'rmsprop_tanh_2x128', 'optimizer': 'rmsprop', 'activation': 'tanh', 'hidden_size': [128, 128]},
    ]
    for cfg in configs:
        args = argparse.Namespace(dataset='fashion_mnist', optimizer=cfg['optimizer'], learning_rate=0.001,
            num_layers=len(cfg['hidden_size']), hidden_size=cfg['hidden_size'], hidden_layers=cfg['hidden_size'],
            activation=cfg['activation'], loss='cross_entropy', weight_init='xavier', weight_decay=0.0)
        run = wandb.init(project='da6401_assignment1', name=f'fashion_{cfg["name"]}', config=cfg)
        model = NeuralNetwork(args)
        for ep in range(epochs):
            avg_loss = model.train(X_train, y_train, epochs=1, batch_size=batch_size)
            res = model.evaluate(X_test, y_test)
            wandb.log({'epoch': ep+1, 'train_loss': avg_loss, 'test_accuracy': res['accuracy']})
        wandb.finish()
        print(f'{cfg["name"]}: accuracy={res["accuracy"]:.4f}')

In [18]:
run_fashion_mnist_3configs(epochs=15)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/ste ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/ste ━━━━━━━━━━━━━━━━━━━━ 1:20 3us/st ━━━━━━━━━━━━━━━━━━━━ 1:49 4us/st ━━━━━━━━━━━━━━━━━━━━ 2:10 5us/st ━━━━━━━━━━━━━━━━━━━━ 1:42 4us/st ━━━━━━━━━━━━━━━━━━━━ 1:38 4us/st ━━━━━━━━━━━━━━━━━━━━ 1:15 3us/st ━━━━━━━━━━━━━━━━━━━━ 1:07 3us/st ━━━━━━━━━━━━━━━━━━━━ 52s 2us/step ━━━━━━━━━━━━━━━━━━━━ 44s 2us/ste ━━━━━━━━━━━━━━━━━━━━ 37s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 39s 2us/ste ━━━━━━━━━━━━━━━━━━━━ 31s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 29s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 29s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 23s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 21s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 21s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 19s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 16s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 16s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 15s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 13s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 12s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 11s 1us/ste ━━━━━━━━━━━━━━━━━━━━ 10s 0us/ste ━━━━━━━━━━━━━━━━━━━━ 9s 0us/ste ━━━━━━━━━━━━━━━━━

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁▆▇▇▆▇▇▇▇▇▇██▇█
train_loss,█▅▄▃▃▃▂▂▂▂▂▁▁▁▁
epoch,15
test_accuracy,0.8828
train_loss,0.2213


rmsprop_relu_3x128: accuracy=0.8828


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁▃▄▅▆▇▇████████
train_loss,██▇▇▆▆▅▄▄▃▃▂▂▁▁
epoch,15
test_accuracy,0.6397
train_loss,1.19552


momentum_relu_3x128: accuracy=0.6397


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁▂▄▄▆▃▆▅▇▇▇█▆▆▆
train_loss,█▅▄▄▃▃▃▂▂▂▂▁▁▁▁
epoch,15
test_accuracy,0.8529
train_loss,0.23738


rmsprop_tanh_2x128: accuracy=0.8529
